<a href="https://colab.research.google.com/github/hedil1/ML-Project/blob/main/Nettoyage_finale_de_3_data.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

 aucune colonne n’a été supprimée afin de préserver un maximum d’information.

Les colonnes contenant un grand nombre de valeurs manquantes ont été transformées en variables indicatrices (présence/absence).

Les valeurs manquantes ont été traitées selon leur nature : médiane pour les variables numériques et "Inconnu" pour les variables catégorielles.



Cette approche permet de maximiser l’information disponible tout en rendant le dataset exploitable par les modèles de Machine Learning.

**DataSet Accidents_SNCFT**
La première base de données concerne les accidents de la SNCFT en 2019. Cette base contient des informations présentées de deux manières différentes. Par exemple, la variable Zone est représentée à la fois sous forme numérique (1, 2, etc.) et sous forme textuelle dans une autre colonne appelée Zone2 (Nord, Est, etc.). Il s’agit donc d’une information dupliquée.

La base contient également deux types de lieux : la pleine voie et la ligne ferroviaire elle-même. On observe par ailleurs de nombreuses valeurs manquantes (NaN), notamment dans les colonnes cause, responsabilité, rapport final, ainsi que d’autres variables.

Concernant la variable source, certaines cellules sont vides ou contiennent uniquement la valeur 0.

In [ ]:
#Import de dataset
import pandas as pd
import numpy as np

df = pd.read_excel("/content/Accidents_SNCFT_2019.xlsx")
df.head()

,N°,n°train,n°machine,Lieu,Lieu.Lieu,Observations,Date,Retard,Nature,Nature incident,...,Obs2,Source,Source.Source,Zone,Zone2,UnitéAffaure,UnitéAffaire,NbVitres,Sécurité,Mois
0,125994,5/87,DMU,B.Arkoub-B.B.Regba,Pleine voie,"DMU827-828-834-835 : Heurt d'une personne, pro...",2019-01-03,28.0,6,Heurt,...,NaN,0,NaN,1,Nord Est,1,Grandes lignes,0.0,0,Janvier
1,126058,13,dp150,manouba-jedeida,Pleine voie,CRE Marzouki-AICRE Manai) : Heurt d'une person...,2019-01-06,19.0,6,Heurt,...,Répercussions : Train 12=47 mn à Jedeida de 13...,0,NaN,2,Nord,1,Grandes lignes,NaN,0,Janvier
2,126084,5/83,gt564,sidi salah-sfax,Pleine voie,"CRE : Mâamri Raouf : Heurt d'une personne, déc...",2019-01-07,58.0,6,Heurt,...,NaN,0,NaN,4,Sud Est,1,Grandes lignes,0.0,0,Janvier
3,126096,22/5/64,dmu,bekalta-moknine,Passage à niveau,PN (BA):DMU831-832-821-822: Tamp.d'une voiture...,2019-01-08,65.0,2,Tamponnement,...,"Rép.Trains B501, B503, B510 et b512 supprimés ...",0,NaN,6,Est,1,Grandes lignes,0.0,0,Janvier
4,126097,6/52,gt556,entrée cheylus,Pleine voie,"Heurt d'un pièton, blessé, transporté par la p...",2019-01-08,45.0,6,Heurt,...,Répercussion : Train 6/51=46 mn à Oudna de 6h3...,0,NaN,1,Nord Est,1,Grandes lignes,0.0,0,Janvier


In [ ]:
#nombre de lignes, nombre de colonnes
df.shape

(97, 30)

In [ ]:
#Info sur Data
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 97 entries, 0 to 96
Data columns (total 30 columns):
 #   Column                      Non-Null Count  Dtype         
---  ------                      --------------  -----         
 0   N°                          97 non-null     int64         
 1   n°train                     97 non-null     object        
 2   n°machine                   97 non-null     object        
 3   Lieu                        97 non-null     object        
 4   Lieu.Lieu                   97 non-null     object        
 5   Observations                97 non-null     object        
 6   Date                        97 non-null     datetime64[ns]
 7   Retard                      91 non-null     float64       
 8   Nature                      97 non-null     int64         
 9   Nature incident             97 non-null     object        
 10  Gare/PK                     97 non-null     int64         
 11  PK Ligne                    96 non-null     float64       
 

In [ ]:
#df.isnull().sum() : Compte le nombre de valeurs manquantes par colonne
#.sort_values(ascending=False) : Trie les colonnes : du plus grand nombre de valeurs manquantes au petit
df.isnull().sum().sort_values(ascending=False)

,0
Source.Source,97
Reponsabilite,96
Cause,96
RapportFinalParvenu,96
RapportPreliminaireParvenu,96
PK ROUTE,94
Obs2,40
Blessés,22
Tués,20
Retard,6


In [ ]:
#compter le nombre de lignes dupliquées dans le DataFrame.
df.duplicated().sum()

np.int64(0)

In [ ]:
#supprimer toutes les lignes dupliquées.
df.duplicated().sum()
df = df.drop_duplicates()

ransformer des colonnes avec beaucoup de valeurs manquantes (sparse columns) en variables indicatrices (features binaires).

```
# Ce texte est au format code
```



In [ ]:
#transformer des colonnes avec beaucoup de valeurs manquantes (sparse columns) en variables indicatrices (features binaires) 1 si info exsite , 0 sinon
cols_sparse = [
    "Cause",
    "Reponsabilite",
    "RapportFinalParvenu",
    "RapportPreliminaireParvenu",
    "PK ROUTE",
    "Source.Source"
]

for col in cols_sparse:
    if col in df.columns:
        df[col + "_exists"] = df[col].notnull().astype(int)

In [ ]:
#Les valeurs manquantes des variables numériques ont été imputées par la médiane afin de limiter l’impact des valeurs aberrantes sur la distribution des données
num_cols = df.select_dtypes(include=["float64", "int64"]).columns

for col in num_cols:
    df[col] = df[col].fillna(df[col].median())

In [ ]:
#traiter les valeurs manquantes dans les colonnes catégorielles (texte) en les remplaçant par une valeur par défaut : "Inconnu"
cat_cols = df.select_dtypes(include=["object"]).columns

for col in cat_cols:
    df[col] = df[col].fillna("Inconnu")

In [ ]:
#nettoyer les données textuelles (colonnes catégorielles) en les rendant uniformes.
for col in cat_cols:
    df[col] = df[col].str.lower().str.strip()

In [ ]:
#créer de nouvelles variables (feature engineering) à partir de colonnes existantes pour mieux exploiter les données dans un modèle ML.
df["obs_length"] = df["Observations"].apply(len)
df["obs2_exists"] = df["Obs2"].notnull().astype(int)
df["pk_route_exists"] = df["PK ROUTE"].notnull().astype(int)

In [ ]:
#Création de la variable “Danger”
df["Danger"] = ((df["Tués"] > 0) | (df["Blessés"] > 0)).astype(int)
#Création de la gravité de l’accident
df["Gravite"] = df["Tués"] * 3 + df["Blessés"] * 2 + df["Retard"] * 0.1
#extrait l’année depuis la date
df["Year"] = df["Date"].dt.year
#extrait le mois
df["Month_num"] = df["Date"].dt.month

transformer les variables catégorielles (texte) en valeurs numériques pour pouvoir les utiliser dans les modèles

In [ ]:
#" LabelEncoder "  outil de scikit-learn qui permet de convertir des catégories en nombres.
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()

for col in cat_cols:
    df[col] = le.fit_transform(df[col].astype(str))

In [ ]:
df.isnull().sum()
df.shape
df.head()

,N°,n°train,n°machine,Lieu,Lieu.Lieu,Observations,Date,Retard,Nature,Nature incident,...,RapportPreliminaireParvenu_exists,PK ROUTE_exists,Source.Source_exists,obs_length,obs2_exists,pk_route_exists,Danger,Gravite,Year,Month_num
0,125994,31,18,3,2,29,2019-01-03,28.0,6,0,...,0,0,0,175,1,1,1,4.8,2019,1
1,126058,7,32,53,2,22,2019-01-06,19.0,6,0,...,0,0,0,249,1,1,1,4.9,2019,1
2,126084,30,54,75,2,13,2019-01-07,58.0,6,0,...,0,0,0,175,1,1,1,8.8,2019,1
3,126096,15,18,11,1,72,2019-01-08,65.0,2,1,...,0,0,0,249,1,1,1,8.5,2019,1
4,126097,40,49,15,2,33,2019-01-08,45.0,6,0,...,0,0,0,142,1,1,1,6.5,2019,1


Nettoayege 3eme data

Le dataset des intersections contenait des noms de gouvernorats en langue arabe ainsi qu’une ligne représentant le total global.

Les noms ont été convertis en codes abrégés standardisés afin d’assurer une meilleure cohérence et compatibilité avec les autres datasets.

La ligne de total a été supprimée car elle ne représente pas une observation réelle.

Les colonnes numériques ont été converties correctement et les valeurs manquantes ont été traitées.

Une nouvelle variable "Total_routes" a été créée pour enrichir l’analyse.

##  Nettoyage du dataset des intersections

Le dataset des intersections a nécessité plusieurs étapes de nettoyage et de préparation afin de garantir sa qualité et sa compatibilité avec les autres sources de données.

###  Standardisation des noms des gouvernorats
Les noms des gouvernorats étaient initialement en langue arabe.  
Ils ont été convertis en codes abrégés standardisés afin d’assurer une cohérence avec les autres datasets et faciliter les opérations de fusion.

###  Suppression des lignes non pertinentes
Une ligne représentant le total global a été identifiée dans le dataset.  
Cette ligne ne correspond pas à une observation réelle, elle a donc été supprimée.

###  Conversion des variables numériques
Certaines colonnes numériques étaient mal typées (format texte).  
Elles ont été converties en format numérique pour permettre leur utilisation dans les analyses et les modèles.

###  Gestion des valeurs manquantes
Les valeurs manquantes ont été traitées en utilisant des méthodes appropriées (remplacement, suppression ou imputation selon le cas).

###  Feature Engineering
Une nouvelle variable appelée **Total_routes** a été créée afin d’enrichir l’analyse et fournir une information supplémentaire sur les infrastructures routières.

In [ ]:
df_inter = pd.read_excel("/content/Intersection_routes_rails.xlsx")

# voir les premières lignes
df_inter.head()

,Les routes,Unnamed: 1,Unnamed: 2,Unnamed: 3,Nbre d'intersection,Gouvernorats
0,Piste,Locales,Régionale,Nationale,NaN,NaN
1,0,0,0,0,0.0,تونس
2,0,0,5,1,6.0,بن عروس
3,6,1,8,1,16.0,منوبة
4,0,0,0,0,0.0,أريانة


In [ ]:
# utiliser la première ligne comme header
df_inter.columns = df_inter.iloc[0]

# supprimer cette ligne
df_inter = df_inter[1:]

In [ ]:
#à renommer les colonnes pour les rendre plus lisibles, cohérentes et faciles à utiliser.
df_inter.columns = [
    "Piste",
    "Locales",
    "Regionale",
    "Nationale",
    "Nb_intersections",
    "Gouvernorat"
]

In [ ]:
df_inter = df_inter.reset_index(drop=True)

Renommer les colonnes propreme

1.   Élément de liste
2.   Élément de liste



In [ ]:
#Renommer les colonnes proprement

df_inter.columns = [
    "Piste",
    "Locales",
    "Regionale",
    "Nationale",
    "Nb_intersections",
    "Gouvernorat"
]

Supprimer la ligne TOTAL

In [ ]:
df_inter = df_inter[df_inter["Gouvernorat"] != "المجموع"]

onversion des noms arabes → codes

In [ ]:
#(mapping) entre les noms des gouvernorats en arabe et leurs codes abrégés.
mapping = {
    "تونس": "TN",
    "بن عروس": "BR",
    "منوبة": "MN",
    "أريانة": "AR",
    "نابل": "NB",
    "المنستير": "MS",
    "المهدية": "MH",
    "سوسة": "SS",
    "صفاقس": "SF",
    "القيروان": "KR",
    "القصرين": "KS",
    "بنزرت": "BZ",
    "تطاوين": "TT",
    "قابس": "GB",
    "مدنين": "MD",
    "زغوان": "ZG",
    "سليانة": "SL",
    "جندوبة": "JN",
    "باجة": "BJ",
    "سيدي بوزيد": "SB",
    "الكاف": "KF",
    "توزر": "TZ",
    "قفصة": "GF",
    "قبلي": "KB"
}

In [ ]:
#Mapping des gouvernorats
df_inter["Gouvernorat"] = df_inter["Gouvernorat"].map(mapping)

In [ ]:
#Vérification des valeurs manquantes
df_inter["Gouvernorat"].isnull().sum()

np.int64(0)

In [ ]:
#Remplacement des valeurs manquantes
df_inter["Gouvernorat"] = df_inter["Gouvernorat"].fillna("UNK")

In [ ]:
#Conversion en numérique des colonnes
cols = ["Piste", "Locales", "Regionale", "Nationale", "Nb_intersections"]

for col in cols:
    df_inter[col] = pd.to_numeric(df_inter[col], errors="coerce")

In [ ]:
#Remplacement global des NaN
df_inter.fillna(0, inplace=True)

In [ ]:
#Création de la variable “Total_routes”
df_inter["Total_routes"] = (
    df_inter["Piste"] +
    df_inter["Locales"] +
    df_inter["Regionale"] +
    df_inter["Nationale"]
)

In [ ]:
#nombre de lignes, nombre de colonnes
df_inter.head()
df_inter.shape

(24, 7)

In [ ]:
#Export du dataset des intersections
df.to_csv("Intersection_routes_railse.csv", index=False)

In [ ]:
#Export du dataset des accidents
df.to_csv("Accidents_SNCFT_2019.csv", index=False)

Data PN

In [ ]:
import pandas as pd

pn = pd.read_csv("PN_SNCFT.csv", encoding='latin-1', sep=';')
acc = pd.read_excel("Accidents_SNCFT_2019.xlsx")
inter = pd.read_excel("Intersection_routes_rails.xlsx")

In [ ]:
#Affichage de 3 dataset
print(" PN")
display(pn.head())

print(" ACCIDENTS ")
display(acc.head())

print("INTERSECTION ")
display(inter.head())

 PN


,PKLigne,Pk1,LIGNE,TYPE,Nbre de Voie,PN voisin gare (à indiquer),TYPE ANI,DE ANP,CDV VPN,ASSAINISSEMENT ZONE ANI,...,Arrondissement,Zone,Latitude,Longitude,Pk Route,Urbain-Rural,Visibilité,Gouvernorat,Institution concernée,Présignalisation
0,"14,750L5","14,75",5,BA,3V,NaN,AF1000,AF1000,2700HZ,BON,...,Bir Kassaa,Zone Nord Est,"36°44'24.36""N","10°19'0.12""E",NaN,NaN,NaN,NaN,NaN,NaN
1,"20,240L5","20,24",5,BA,2V,NaN,AF1000,AF1000,12+15KHZ,BON,...,Bir Kassaa,Zone Nord Est,"36°42'50.35""N","10°22'6.89""E",NaN,NaN,NaN,NaN,NaN,NaN
2,"4,600L6","4,6",6,BA,3V,NaN,Ple Electronique,Ple Electronique,8700H+Ple ELEC,NEANT,...,Bir Kassaa,Zone Nord Est,"36°46'6.76""N","10°12'56.22""E",NaN,NaN,NaN,NaN,NaN,NaN
3,"3,973LTA","3,973",TA,BA,2V,,Courant continu,Courant continu,Courant continu,BON,...,Bir Kassaa,Zone Nord Est,"36°47'52.10""N","10° 9'12.19""E",NaN,NaN,NaN,NaN,NaN,NaN
4,"4,360LTA","4,36",TA,BA,2V,,Courant continu,Courant continu,Courant continu,BON,...,Bir Kassaa,Zone Nord Est,"36°48'5.03""N","10° 8'53.07""E",NaN,NaN,NaN,NaN,NaN,NaN


 ACCIDENTS 


,N°,n°train,n°machine,Lieu,Lieu.Lieu,Observations,Date,Retard,Nature,Nature incident,...,Obs2,Source,Source.Source,Zone,Zone2,UnitéAffaure,UnitéAffaire,NbVitres,Sécurité,Mois
0,125994,5/87,DMU,B.Arkoub-B.B.Regba,Pleine voie,"DMU827-828-834-835 : Heurt d'une personne, pro...",2019-01-03,28.0,6,Heurt,...,NaN,0,NaN,1,Nord Est,1,Grandes lignes,0.0,0,Janvier
1,126058,13,dp150,manouba-jedeida,Pleine voie,CRE Marzouki-AICRE Manai) : Heurt d'une person...,2019-01-06,19.0,6,Heurt,...,Répercussions : Train 12=47 mn à Jedeida de 13...,0,NaN,2,Nord,1,Grandes lignes,NaN,0,Janvier
2,126084,5/83,gt564,sidi salah-sfax,Pleine voie,"CRE : Mâamri Raouf : Heurt d'une personne, déc...",2019-01-07,58.0,6,Heurt,...,NaN,0,NaN,4,Sud Est,1,Grandes lignes,0.0,0,Janvier
3,126096,22/5/64,dmu,bekalta-moknine,Passage à niveau,PN (BA):DMU831-832-821-822: Tamp.d'une voiture...,2019-01-08,65.0,2,Tamponnement,...,"Rép.Trains B501, B503, B510 et b512 supprimés ...",0,NaN,6,Est,1,Grandes lignes,0.0,0,Janvier
4,126097,6/52,gt556,entrée cheylus,Pleine voie,"Heurt d'un pièton, blessé, transporté par la p...",2019-01-08,45.0,6,Heurt,...,Répercussion : Train 6/51=46 mn à Oudna de 6h3...,0,NaN,1,Nord Est,1,Grandes lignes,0.0,0,Janvier


INTERSECTION 


,Les routes,Unnamed: 1,Unnamed: 2,Unnamed: 3,Nbre d'intersection,Gouvernorats
0,Piste,Locales,Régionale,Nationale,NaN,NaN
1,0,0,0,0,0.0,تونس
2,0,0,5,1,6.0,بن عروس
3,6,1,8,1,16.0,منوبة
4,0,0,0,0,0.0,أريانة


In [ ]:
#details lign e / colones d echaque data
print("PN:", pn.shape)
print("ACC:", acc.shape)
print("INTER:", inter.shape)

PN: (1134, 38)
ACC: (97, 30)
INTER: (26, 6)


In [ ]:
#details de chaque data
pn.info()
acc.info()
inter.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1134 entries, 0 to 1133
Data columns (total 38 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   PKLigne                       1134 non-null   object 
 1   Pk1                           1134 non-null   object 
 2   LIGNE                         1134 non-null   object 
 3    TYPE                         1134 non-null   object 
 4   Nbre de Voie                  296 non-null    object 
 5   PN voisin gare (à indiquer)   137 non-null    object 
 6          TYPE ANI               288 non-null    object 
 7   DE ANP                        288 non-null    object 
 8   CDV VPN                       288 non-null    object 
 9   ASSAINISSEMENT ZONE ANI       289 non-null    object 
 10  ASSAINISSEMENT ZONE ANP       288 non-null    object 
 11  ASSAINISSEMENT ZONE VPN       288 non-null    object 
 12  CONTACT RAIL ANI              289 non-null    object 
 13  CON

In [ ]:
#Valeurs manquantes de chaque data
print(pn.isnull().sum())
print(acc.isnull().sum())
print(inter.isnull().sum())

PKLigne                            0
Pk1                                0
LIGNE                              0
 TYPE                              0
Nbre de Voie                     838
PN voisin gare (à indiquer)      997
       TYPE ANI                  846
DE ANP                           846
CDV VPN                          846
ASSAINISSEMENT ZONE ANI          845
ASSAINISSEMENT ZONE ANP          846
ASSAINISSEMENT ZONE VPN          846
CONTACT RAIL ANI                 845
CONTACT RAIL ANP                 846
CONTACT RAIL VPN                 846
DUREE ANNONCE  PAIRE             845
DUREE ANNONCE IMPAIRE            845
terre                            845
coupure bipolaire                845
PN avec clignoteur à mercure     845
PN dôté d'alim commune           845
Etat du câblage châssis          845
Etat de la guérite               845
PN dôté de minuterie             845
type Télécontrôle                845
ETAT\n BATTERIE                  845
OBSERVATIONS                     922
O

Nettoayege de data PN : ## Sélection des variables

Le dataset initial contient 38 colonnes, dont plusieurs variables techniques non pertinentes pour notre problème de prédiction.

Nous avons sélectionné les variables les plus importantes :

Caractéristiques structurelles (TYPE, nombre de voies)
Localisation (zone, gouvernorat)
Facteurs de risque (visibilité, urbain/rural)
Coordonnées géographiques
Les autres variables ont été supprimées car elles contiennent beaucoup de valeurs manquantes ou ne sont pas directement liées au risque d'accident.

In [ ]:
#nettoyer les noms des colonnes dans les trois Data en supprimant les espaces inutiles.
pn.columns = pn.columns.str.strip()
acc.columns = acc.columns.str.strip()
inter.columns = inter.columns.str.strip()

In [ ]:
pn = pn[[
    'PKLigne',
    'TYPE',
    'Nbre de Voie',
    'Arrondissement',
    'Zone',
    'Latitude',
    'Longitude',
    'Urbain-Rural',
    'Visibilité',
    'Gouvernorat',
    'Présignalisation'
]]

In [ ]:
#détecter et compter les valeurs manquantes (NaN) dans le Data PN
pn.isnull().sum()

,0
PKLigne,0
TYPE,0
Nbre de Voie,838
Arrondissement,837
Zone,837
Latitude,5
Longitude,6
Urbain-Rural,1134
Visibilité,1134
Gouvernorat,1134


Corriger les données manquantes :

Les variables catégorielles ont été complétées avec la valeur "Inconnu"
Les variables numériques ont été complétées par la valeur la plus fréquente
Les coordonnées géographiques ont été complétées par propagation (forward fill)

In [ ]:
#les valeurs manquantes dans des colonnes catégorielles du dataset PN en les remplaçant par "Inconnu".
pn['Visibilité'] = pn['Visibilité'].fillna('Inconnu')
pn['Gouvernorat'] = pn['Gouvernorat'].fillna('Inconnu')
pn['Présignalisation'] = pn['Présignalisation'].fillna('Inconnu')

In [ ]:
#remplacer les valeurs manquantes dans une variable catégorielle/numérique par la valeur la plus fréquente (mode)
pn['Nbre de Voie'] = pn['Nbre de Voie'].fillna(pn['Nbre de Voie'].mode()[0])

In [ ]:
#traiter les valeurs manquantes dans des colonnes catégorielles du dataset pn en les remplaçant par "Inconnu".
pn['Arrondissement'] = pn['Arrondissement'].fillna('Inconnu')
pn['Zone'] = pn['Zone'].fillna('Inconnu')

In [ ]:
#remplacer les valeurs manquantes dans les colonnes géographiques (Latitude et Longitude) en utilisant la méthode forward fill (propagation vers l’avant).
pn['Latitude'] = pn['Latitude'].fillna(method='ffill')
pn['Longitude'] = pn['Longitude'].fillna(method='ffill')

/tmp/ipykernel_13944/1838221764.py:1: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  pn['Latitude'] = pn['Latitude'].fillna(method='ffill')
/tmp/ipykernel_13944/1838221764.py:2: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  pn['Longitude'] = pn['Longitude'].fillna(method='ffill')


In [ ]:
print(acc.columns)
acc.isnull().sum()

Index(['N°', 'n°train', 'n°machine', 'Lieu', 'Lieu.Lieu', 'Observations',
       'Date', 'Retard', 'Nature', 'Nature incident', 'Gare/PK', 'PK Ligne',
       'PK ROUTE', 'Blessés', 'Tués', 'Ligne', 'Cause', 'Reponsabilite',
       'RapportPreliminaireParvenu', 'RapportFinalParvenu', 'Obs2', 'Source',
       'Source.Source', 'Zone', 'Zone2', 'UnitéAffaure', 'UnitéAffaire',
       'NbVitres', 'Sécurité', 'Mois'],
      dtype='object')


,0
N°,0
n°train,0
n°machine,0
Lieu,0
Lieu.Lieu,0
Observations,0
Date,0
Retard,6
Nature,0
Nature incident,0


In [ ]:
print("DATASET ACCIDENTS")
df.head()

===== DATASET ACCIDENTS =====


,N°,n°train,n°machine,Lieu,Lieu.Lieu,Observations,Date,Retard,Nature,Nature incident,...,RapportPreliminaireParvenu_exists,PK ROUTE_exists,Source.Source_exists,obs_length,obs2_exists,pk_route_exists,Danger,Gravite,Year,Month_num
0,125994,31,18,3,2,29,2019-01-03,28.0,6,0,...,0,0,0,175,1,1,1,4.8,2019,1
1,126058,7,32,53,2,22,2019-01-06,19.0,6,0,...,0,0,0,249,1,1,1,4.9,2019,1
2,126084,30,54,75,2,13,2019-01-07,58.0,6,0,...,0,0,0,175,1,1,1,8.8,2019,1
3,126096,15,18,11,1,72,2019-01-08,65.0,2,1,...,0,0,0,249,1,1,1,8.5,2019,1
4,126097,40,49,15,2,33,2019-01-08,45.0,6,0,...,0,0,0,142,1,1,1,6.5,2019,1


In [ ]:
print("DATASET INTERSECTION ")
df_inter.head()

===== DATASET INTERSECTION =====


,Piste,Locales,Regionale,Nationale,Nb_intersections,Gouvernorat,Total_routes
0,0,0,0,0,0.0,TN,0
1,0,0,5,1,6.0,BR,6
2,6,1,8,1,16.0,MN,16
3,0,0,0,0,0.0,AR,0
4,3,1,1,6,11.0,NB,11


Affichage apres nettoayage

In [ ]:
#Details apres nettoayeg
print("DATASET PN ")
display(pn.head())

print(" DATASET ACCIDENTs")
display(acc.head())

print(" DATASET INTERSECTIONS")
display(inter.head())

=== DATASET PN ===


,PKLigne,TYPE,Nbre de Voie,Arrondissement,Zone,Latitude,Longitude,Urbain-Rural,Visibilité,Gouvernorat,Présignalisation
0,"14,750L5",BA,3V,Bir Kassaa,Zone Nord Est,"36°44'24.36""N","10°19'0.12""E",Inconnu,Inconnu,Inconnu,Inconnu
1,"20,240L5",BA,2V,Bir Kassaa,Zone Nord Est,"36°42'50.35""N","10°22'6.89""E",Inconnu,Inconnu,Inconnu,Inconnu
2,"4,600L6",BA,3V,Bir Kassaa,Zone Nord Est,"36°46'6.76""N","10°12'56.22""E",Inconnu,Inconnu,Inconnu,Inconnu
3,"3,973LTA",BA,2V,Bir Kassaa,Zone Nord Est,"36°47'52.10""N","10° 9'12.19""E",Inconnu,Inconnu,Inconnu,Inconnu
4,"4,360LTA",BA,2V,Bir Kassaa,Zone Nord Est,"36°48'5.03""N","10° 8'53.07""E",Inconnu,Inconnu,Inconnu,Inconnu



=== DATASET ACCIDENTS ===


,N°,n°train,n°machine,Lieu,Lieu.Lieu,Observations,Date,Retard,Nature,Nature incident,...,Obs2,Source,Source.Source,Zone,Zone2,UnitéAffaure,UnitéAffaire,NbVitres,Sécurité,Mois
0,125994,5/87,DMU,B.Arkoub-B.B.Regba,Pleine voie,"DMU827-828-834-835 : Heurt d'une personne, pro...",2019-01-03,28.0,6,Heurt,...,NaN,0,NaN,1,Nord Est,1,Grandes lignes,0.0,0,Janvier
1,126058,13,dp150,manouba-jedeida,Pleine voie,CRE Marzouki-AICRE Manai) : Heurt d'une person...,2019-01-06,19.0,6,Heurt,...,Répercussions : Train 12=47 mn à Jedeida de 13...,0,NaN,2,Nord,1,Grandes lignes,NaN,0,Janvier
2,126084,5/83,gt564,sidi salah-sfax,Pleine voie,"CRE : Mâamri Raouf : Heurt d'une personne, déc...",2019-01-07,58.0,6,Heurt,...,NaN,0,NaN,4,Sud Est,1,Grandes lignes,0.0,0,Janvier
3,126096,22/5/64,dmu,bekalta-moknine,Passage à niveau,PN (BA):DMU831-832-821-822: Tamp.d'une voiture...,2019-01-08,65.0,2,Tamponnement,...,"Rép.Trains B501, B503, B510 et b512 supprimés ...",0,NaN,6,Est,1,Grandes lignes,0.0,0,Janvier
4,126097,6/52,gt556,entrée cheylus,Pleine voie,"Heurt d'un pièton, blessé, transporté par la p...",2019-01-08,45.0,6,Heurt,...,Répercussion : Train 6/51=46 mn à Oudna de 6h3...,0,NaN,1,Nord Est,1,Grandes lignes,0.0,0,Janvier



=== DATASET INTERSECTIONS ===


,Les routes,Unnamed: 1,Unnamed: 2,Unnamed: 3,Nbre d'intersection,Gouvernorats
0,Piste,Locales,Régionale,Nationale,NaN,NaN
1,0,0,0,0,0.0,تونس
2,0,0,5,1,6.0,بن عروس
3,6,1,8,1,16.0,منوبة
4,0,0,0,0,0.0,أريانة


In [ ]:
#dimensions des datasets
print("Dimensions :")
print("PN :", pn.shape)
print("ACC :", acc.shape)
print("INTER :", inter.shape)

Dimensions :
PN : (1134, 11)
ACC : (97, 30)
INTER : (26, 6)


In [ ]:
#Export  en Excel
pn.to_excel("PN_clean.xlsx", index=False)
acc.to_excel("ACC_clean.xlsx", index=False)
inter.to_excel("INTER_clean.xlsx", index=False)

In [ ]:
##Export  en Excel
with pd.ExcelWriter("datasets_clean.xlsx") as writer:
    pn.to_excel(writer, sheet_name="PN", index=False)
    acc.to_excel(writer, sheet_name="ACCIDENTS", index=False)
    inter.to_excel(writer, sheet_name="INTERSECTIONS", index=False)

print(" Fichier Excel créé avec succès")

 Fichier Excel créé avec succès


In [ ]:
from google.colab import files
files.download("datasets_clean.xlsx")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

2eme Nettoyage de Data PN car 1er nettaoyg n'st pas bien Plusieurs étapes de nettoyage ont été réalisées :

- Conversion des coordonnées géographiques (Latitude, Longitude) en format numérique exploitable
- Normalisation de la variable "Nombre de voie"
- Remplacement des valeurs "Inconnu" par des valeurs manquantes ou catégories standardisées
- Extraction des informations pertinentes à partir de la variable PKLigne

Ces transformations permettent d'améliorer la qualité des données et de les rendre exploitables pour les modèles de Machine Learning.

In [ ]:
#convertir les coordonnées géographiques (format degrés/minutes/secondes) en coordonnées décimales
import re   #la bibliothèque regex travailler avec les expressions régulières pour extraire les parties des coordonnées.

def convert_coord(coord): #Fonction qui transforme une coordonnée texte en nombre décimal.

    if pd.isna(coord):  #Gestion des valeurs manquantes
        return None
    coord = coord.replace(" ", "") #Nettoyage de la chaîne
    match = re.match(r"(\d+)°(\d+)'([\d\.]+)\"([NSEW])", coord) #Extraction avec expression régulière degre , minu; secon ; Directeion N, S, E , W
    if match:
        deg, minutes, sec, direction = match.groups()
        decimal = float(deg) + float(minutes)/60 + float(sec)/3600 #Conversion en format décimal
        if direction in ['S', 'W']:
            decimal = -decimal
        return decimal
    return None

pn['Latitude'] = pn['Latitude'].apply(convert_coord)
pn['Longitude'] = pn['Longitude'].apply(convert_coord)

In [ ]:
#extraire et transformer la variable “Nbre de Voie” en une valeur numérique propre
pn['Nbre de Voie'] = pn['Nbre de Voie'].str.extract('(\d+)')
pn['Nbre de Voie'] = pd.to_numeric(pn['Nbre de Voie'], errors='coerce')

<>:1: SyntaxWarning: invalid escape sequence '\d'
<>:1: SyntaxWarning: invalid escape sequence '\d'
/tmp/ipykernel_13944/1163451424.py:1: SyntaxWarning: invalid escape sequence '\d'
  pn['Nbre de Voie'] = pn['Nbre de Voie'].str.extract('(\d+)')


In [ ]:
pn.replace("Inconnu", pd.NA, inplace=True)

In [ ]:
pn = pn.fillna({
    'Urbain-Rural': 'Unknown',
    'Visibilité': 'Unknown',
    'Gouvernorat': 'Unknown',
    'Présignalisation': 'Unknown'
})

In [ ]:
pn['PK'] = pn['PKLigne'].str.extract('(\d+,\d+)')
pn['PK'] = pn['PK'].str.replace(',', '.').astype(float)

pn['Ligne'] = pn['PKLigne'].str.extract('(L\d+)')

<>:1: SyntaxWarning: invalid escape sequence '\d'
<>:4: SyntaxWarning: invalid escape sequence '\d'
<>:1: SyntaxWarning: invalid escape sequence '\d'
<>:4: SyntaxWarning: invalid escape sequence '\d'
/tmp/ipykernel_13944/272537713.py:1: SyntaxWarning: invalid escape sequence '\d'
  pn['PK'] = pn['PKLigne'].str.extract('(\d+,\d+)')
/tmp/ipykernel_13944/272537713.py:4: SyntaxWarning: invalid escape sequence '\d'
  pn['Ligne'] = pn['PKLigne'].str.extract('(L\d+)')


In [ ]:
print(pn.info())
display(pn.head())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1134 entries, 0 to 1133
Data columns (total 13 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   PKLigne           1134 non-null   object 
 1   TYPE              1134 non-null   object 
 2   Nbre de Voie      1000 non-null   float64
 3   Arrondissement    297 non-null    object 
 4   Zone              297 non-null    object 
 5   Latitude          1133 non-null   float64
 6   Longitude         1134 non-null   float64
 7   Urbain-Rural      1134 non-null   object 
 8   Visibilité        1134 non-null   object 
 9   Gouvernorat       1134 non-null   object 
 10  Présignalisation  1134 non-null   object 
 11  PK                402 non-null    float64
 12  Ligne             999 non-null    object 
dtypes: float64(4), object(9)
memory usage: 115.3+ KB
None


,PKLigne,TYPE,Nbre de Voie,Arrondissement,Zone,Latitude,Longitude,Urbain-Rural,Visibilité,Gouvernorat,Présignalisation,PK,Ligne
0,"14,750L5",BA,3.0,Bir Kassaa,Zone Nord Est,36.740100,10.316700,Unknown,Unknown,Unknown,Unknown,14.750,L5
1,"20,240L5",BA,2.0,Bir Kassaa,Zone Nord Est,36.713986,10.368581,Unknown,Unknown,Unknown,Unknown,20.240,L5
2,"4,600L6",BA,3.0,Bir Kassaa,Zone Nord Est,36.768544,10.215617,Unknown,Unknown,Unknown,Unknown,4.600,L6
3,"3,973LTA",BA,2.0,Bir Kassaa,Zone Nord Est,36.797806,10.153386,Unknown,Unknown,Unknown,Unknown,3.973,NaN
4,"4,360LTA",BA,2.0,Bir Kassaa,Zone Nord Est,36.801397,10.148075,Unknown,Unknown,Unknown,Unknown,4.360,NaN


In [ ]:
import pandas as pd

# Charger les données
df = pd.read_excel("Accidents_SNCFT_2019.xlsx")
# Colonnes à supprimer
cols_to_drop = [
    "PK ROUTE",
    "Cause",
    "Reponsabilite",
    "RapportPreliminaireParvenu",
    "RapportFinalParvenu"
]

# Suppression des colonnes
df_clean = df.drop(columns=cols_to_drop, errors='ignore')

# Vérification
print(df_clean.columns)

# Sauvegarde du dataset nettoyé
df_clean.to_csv("accidents_clean.csv", index=False)

Index(['N°', 'n°train', 'n°machine', 'Lieu', 'Lieu.Lieu', 'Observations',
       'Date', 'Retard', 'Nature', 'Nature incident', 'Gare/PK', 'PK Ligne',
       'Blessés', 'Tués', 'Ligne', 'Obs2', 'Source', 'Source.Source', 'Zone',
       'Zone2', 'UnitéAffaure', 'UnitéAffaire', 'NbVitres', 'Sécurité',
       'Mois'],
      dtype='object')


In [ ]:
df_clean = df_clean.replace(["Inconnu", "NA", ""], pd.NA)

In [ ]:
print(df.columns)

Index(['N°', 'n°train', 'n°machine', 'Lieu', 'Lieu.Lieu', 'Observations',
       'Date', 'Retard', 'Nature', 'Nature incident', 'Gare/PK', 'PK Ligne',
       'PK ROUTE', 'Blessés', 'Tués', 'Ligne', 'Cause', 'Reponsabilite',
       'RapportPreliminaireParvenu', 'RapportFinalParvenu', 'Obs2', 'Source',
       'Source.Source', 'Zone', 'Zone2', 'UnitéAffaure', 'UnitéAffaire',
       'NbVitres', 'Sécurité', 'Mois'],
      dtype='object')


In [ ]:
df.columns = df.columns.str.strip()

In [ ]:
cols_to_drop = [
    "PK ROUTE",
    "Cause",
    "Reponsabilite",
    "RapportPreliminaireParvenu",
    "RapportFinalParvenu"
]

df_clean = df.drop(columns=cols_to_drop, errors="ignore")

In [ ]:
df_clean.head()

,N°,n°train,n°machine,Lieu,Lieu.Lieu,Observations,Date,Retard,Nature,Nature incident,...,Obs2,Source,Source.Source,Zone,Zone2,UnitéAffaure,UnitéAffaire,NbVitres,Sécurité,Mois
0,125994,5/87,DMU,B.Arkoub-B.B.Regba,Pleine voie,"DMU827-828-834-835 : Heurt d'une personne, pro...",2019-01-03,28.0,6,Heurt,...,NaN,0,NaN,1,Nord Est,1,Grandes lignes,0.0,0,Janvier
1,126058,13,dp150,manouba-jedeida,Pleine voie,CRE Marzouki-AICRE Manai) : Heurt d'une person...,2019-01-06,19.0,6,Heurt,...,Répercussions : Train 12=47 mn à Jedeida de 13...,0,NaN,2,Nord,1,Grandes lignes,NaN,0,Janvier
2,126084,5/83,gt564,sidi salah-sfax,Pleine voie,"CRE : Mâamri Raouf : Heurt d'une personne, déc...",2019-01-07,58.0,6,Heurt,...,NaN,0,NaN,4,Sud Est,1,Grandes lignes,0.0,0,Janvier
3,126096,22/5/64,dmu,bekalta-moknine,Passage à niveau,PN (BA):DMU831-832-821-822: Tamp.d'une voiture...,2019-01-08,65.0,2,Tamponnement,...,"Rép.Trains B501, B503, B510 et b512 supprimés ...",0,NaN,6,Est,1,Grandes lignes,0.0,0,Janvier
4,126097,6/52,gt556,entrée cheylus,Pleine voie,"Heurt d'un pièton, blessé, transporté par la p...",2019-01-08,45.0,6,Heurt,...,Répercussion : Train 6/51=46 mn à Oudna de 6h3...,0,NaN,1,Nord Est,1,Grandes lignes,0.0,0,Janvier


In [ ]:
df_clean.to_csv("accidents_clean.csv", index=False)

In [ ]:
from google.colab import files

files.download("accidents_clean.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# 1. Sauvegarde
df_clean.to_excel("accidents_clean.xlsx", index=False)

# 2. Téléchargement
from google.colab import files
files.download("accidents_clean.xlsx")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
cols_to_drop = [
    "n°machine",
    "n°train",
    "Nature",
    "Nature incident",
    "Source",
    "Source.Source",
    "Zone",
    "NbVitres"
]

df_clean = df.drop(columns=cols_to_drop, errors="ignore")

In [ ]:
print(df_clean.columns)
df_clean.head()

Index(['N°', 'Lieu', 'Lieu.Lieu', 'Observations', 'Date', 'Retard', 'Gare/PK',
       'PK Ligne', 'PK ROUTE', 'Blessés', 'Tués', 'Ligne', 'Cause',
       'Reponsabilite', 'RapportPreliminaireParvenu', 'RapportFinalParvenu',
       'Obs2', 'Zone2', 'UnitéAffaure', 'UnitéAffaire', 'Sécurité', 'Mois'],
      dtype='object')


,N°,Lieu,Lieu.Lieu,Observations,Date,Retard,Gare/PK,PK Ligne,PK ROUTE,Blessés,...,Cause,Reponsabilite,RapportPreliminaireParvenu,RapportFinalParvenu,Obs2,Zone2,UnitéAffaure,UnitéAffaire,Sécurité,Mois
0,125994,B.Arkoub-B.B.Regba,Pleine voie,"DMU827-828-834-835 : Heurt d'une personne, pro...",2019-01-03,28.0,2,51.200001,NaN,1.0,...,NaN,NaN,NaN,NaN,NaN,Nord Est,1,Grandes lignes,0,Janvier
1,126058,manouba-jedeida,Pleine voie,CRE Marzouki-AICRE Manai) : Heurt d'une person...,2019-01-06,19.0,2,17.900000,NaN,NaN,...,NaN,NaN,NaN,NaN,Répercussions : Train 12=47 mn à Jedeida de 13...,Nord,1,Grandes lignes,0,Janvier
2,126084,sidi salah-sfax,Pleine voie,"CRE : Mâamri Raouf : Heurt d'une personne, déc...",2019-01-07,58.0,2,258.000000,NaN,0.0,...,NaN,NaN,NaN,NaN,NaN,Sud Est,1,Grandes lignes,0,Janvier
3,126096,bekalta-moknine,Passage à niveau,PN (BA):DMU831-832-821-822: Tamp.d'une voiture...,2019-01-08,65.0,4,43.720001,NaN,1.0,...,NaN,NaN,NaN,NaN,"Rép.Trains B501, B503, B510 et b512 supprimés ...",Est,1,Grandes lignes,0,Janvier
4,126097,entrée cheylus,Pleine voie,"Heurt d'un pièton, blessé, transporté par la p...",2019-01-08,45.0,2,35.299999,NaN,1.0,...,NaN,NaN,NaN,NaN,Répercussion : Train 6/51=46 mn à Oudna de 6h3...,Nord Est,1,Grandes lignes,0,Janvier


In [ ]:
df_clean.to_excel("accidents_clean_final.xlsx", index=False)

In [ ]:
from google.colab import files
files.download("accidents_clean_final.xlsx")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
cols_to_drop = [
    # anciennes colonnes
    "n°machine",
    "n°train",
    "Nature",
    "Nature incident",
    "Source",
    "Source.Source",
    "Zone",
    "NbVitres",

    # nouvelles colonnes
    "PK ROUTE",
    "Cause",
    "Reponsabilite",
    "RapportPreliminaireParvenu",
    "RapportFinalParvenu",
    "Observations"
]

df_clean = df.drop(columns=cols_to_drop, errors="ignore")

In [ ]:
print(df_clean.columns)
df_clean.head()

Index(['N°', 'Lieu', 'Lieu.Lieu', 'Date', 'Retard', 'Gare/PK', 'PK Ligne',
       'Blessés', 'Tués', 'Ligne', 'Obs2', 'Zone2', 'UnitéAffaure',
       'UnitéAffaire', 'Sécurité', 'Mois'],
      dtype='object')


,N°,Lieu,Lieu.Lieu,Date,Retard,Gare/PK,PK Ligne,Blessés,Tués,Ligne,Obs2,Zone2,UnitéAffaure,UnitéAffaire,Sécurité,Mois
0,125994,B.Arkoub-B.B.Regba,Pleine voie,2019-01-03,28.0,2,51.200001,1.0,0.0,5,NaN,Nord Est,1,Grandes lignes,0,Janvier
1,126058,manouba-jedeida,Pleine voie,2019-01-06,19.0,2,17.900000,NaN,1.0,13,Répercussions : Train 12=47 mn à Jedeida de 13...,Nord,1,Grandes lignes,0,Janvier
2,126084,sidi salah-sfax,Pleine voie,2019-01-07,58.0,2,258.000000,0.0,1.0,5,NaN,Sud Est,1,Grandes lignes,0,Janvier
3,126096,bekalta-moknine,Passage à niveau,2019-01-08,65.0,4,43.720001,1.0,0.0,22,"Rép.Trains B501, B503, B510 et b512 supprimés ...",Est,1,Grandes lignes,0,Janvier
4,126097,entrée cheylus,Pleine voie,2019-01-08,45.0,2,35.299999,1.0,0.0,6,Répercussion : Train 6/51=46 mn à Oudna de 6h3...,Nord Est,1,Grandes lignes,0,Janvier


In [ ]:
df_clean.to_excel("accidents_clean_final.xlsx", index=False)

In [ ]:
from google.colab import files
files.download("accidents_clean_final.xlsx")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
df_clean["Retard_missing"] = df_clean["Retard"].isnull().astype(int)
df_clean["Blessés_missing"] = df_clean["Blessés"].isnull().astype(int)
df_clean["Tués_missing"] = df_clean["Tués"].isnull().astype(int)

In [ ]:
df_clean["Retard"] = df_clean["Retard"].fillna(df_clean["Retard"].median())
df_clean["Blessés"] = df_clean["Blessés"].fillna(df_clean["Blessés"].median())
df_clean["Tués"] = df_clean["Tués"].fillna(df_clean["Tués"].median())

In [ ]:
df_clean = df_clean.drop(columns=["Obs2"], errors="ignore")

In [ ]:
df_clean.isnull().sum()

,0
N°,0
Lieu,0
Lieu.Lieu,0
Date,0
Retard,0
Gare/PK,0
PK Ligne,1
Blessés,0
Tués,0
Ligne,0


In [ ]:
df_clean.to_csv("accidents_clean_final.csv", index=False)

In [ ]:
df_clean.to_excel("accidents_clean_final.xlsx", index=False)

In [ ]:
from google.colab import files

files.download("accidents_clean_final.csv")
files.download("accidents_clean_final.xlsx")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
%who

LabelEncoder	 acc	 cat_cols	 col	 cols	 cols_sparse	 cols_to_drop	 convert_coord	 df	 
df_clean	 df_inter	 files	 inter	 le	 mapping	 np	 num_cols	 pd	 
pn	 re	 writer	 


In [ ]:
routes = df_inter.copy()

In [ ]:
print(routes.columns)

Index(['Piste', 'Locales', 'Regionale', 'Nationale', 'Nb_intersections',
       'Gouvernorat', 'Total_routes'],
      dtype='object')


In [ ]:
routes = routes[routes["Gouvernorat"] != "المجموع"]

In [ ]:
mapping = {
    "تونس": "TN",
    "بن عروس": "BN",
    "منوبة": "MN",
    "أريانة": "AR",
    "نابل": "NB",
    "المنستير": "MS",
    "المهدية": "MD",
    "سوسة": "SS",
    "صفاقس": "SF",
    "القيروان": "KR",
    "القصرين": "KS",
    "بنزرت": "BZ",
    "تطاوين": "TT",
    "قابس": "GB",
    "مدنين": "MDN",
    "زغوان": "ZG",
    "سليانة": "SL",
    "جندوبة": "JN",
    "باجة": "BJ",
    "سيدي بوزيد": "SB",
    "الكاف": "KF",
    "توزر": "TZ",
    "قفصة": "GF",
    "قبلي": "KB"
}

routes["Gouvernorat"] = routes["Gouvernorat"].map(mapping)

In [ ]:
print(routes.head())
print(routes.isnull().sum())

   Piste  Locales  Regionale  Nationale  Nb_intersections Gouvernorat  \
0      0        0          0          0               0.0         NaN   
1      0        0          5          1               6.0         NaN   
2      6        1          8          1              16.0         NaN   
3      0        0          0          0               0.0         NaN   
4      3        1          1          6              11.0         NaN   

   Total_routes  
0             0  
1             6  
2            16  
3             0  
4            11  
Piste                0
Locales              0
Regionale            0
Nationale            0
Nb_intersections     0
Gouvernorat         24
Total_routes         0
dtype: int64


In [ ]:
routes.to_excel("routes_clean.xlsx", index=False)

In [ ]:
from google.colab import files
files.download("routes_clean.xlsx")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
print(routes["Gouvernorat"].unique())

[nan]


In [ ]:
routes = df_inter.copy()

In [ ]:
print(routes["Gouvernorat"].unique())

['TN' 'BR' 'MN' 'AR' 'NB' 'MS' 'MH' 'SS' 'SF' 'KR' 'KS' 'BZ' 'TT' 'GB'
 'MD' 'ZG' 'SL' 'JN' 'BJ' 'SB' 'KF' 'TZ' 'GF' 'KB']


In [ ]:
routes = df_inter.copy()

routes["Gouvernorat"] = routes["Gouvernorat"].astype(str).str.strip()

mapping = {
    "تونس": "TN",
    "بن عروس": "BN",
    "منوبة": "MN",
    "أريانة": "AR",
    "نابل": "NB",
    "المنستير": "MS",
    "المهدية": "MD",
    "سوسة": "SS",
    "صفاقس": "SF",
    "القيروان": "KR",
    "القصرين": "KS",
    "بنزرت": "BZ",
    "تطاوين": "TT",
    "قابس": "GB",
    "مدنين": "MDN",
    "زغوان": "ZG",
    "سليانة": "SL",
    "جندوبة": "JN",
    "باجة": "BJ",
    "سيدي بوزيد": "SB",
    "الكاف": "KF",
    "توزر": "TZ",
    "قفصة": "GF",
    "قبلي": "KB"
}

routes["Gouvernorat"] = routes["Gouvernorat"].map(mapping).fillna(routes["Gouvernorat"])

In [ ]:
print(routes["Gouvernorat"].unique())

['TN' 'BR' 'MN' 'AR' 'NB' 'MS' 'MH' 'SS' 'SF' 'KR' 'KS' 'BZ' 'TT' 'GB'
 'MD' 'ZG' 'SL' 'JN' 'BJ' 'SB' 'KF' 'TZ' 'GF' 'KB']


In [ ]:
routes.to_excel("routes_avant_mapping.xlsx", index=False)

In [ ]:
from google.colab import files
files.download("routes_avant_mapping.xlsx")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>